In [1]:
import numpy as np
import pandas as pd

# Reproducibility
np.random.seed(42)

# Number of transactions
n = 5000

# -----------------------------
# 1. Basic transaction details
# -----------------------------

transaction_id = np.arange(100001, 100001 + n)

timestamp = pd.date_range(
    start="2025-01-01",
    end="2025-12-31 23:59:59",
    periods=n
)

user_id = np.random.randint(1001, 2001, n)
receiver_id = np.random.randint(5001, 6001, n)

# Transaction amount
amount = np.round(
    np.random.lognormal(mean=6.2, sigma=1.0, size=n),
    2
)

amount = np.clip(amount, 10, 100000)

# -----------------------------
# 2. Categorical features
# -----------------------------

transaction_type = np.random.choice(
    ["send", "receive", "merchant_payment"],
    size=n,
    p=[0.45, 0.20, 0.35]
)

location = np.random.choice(
    ["Mumbai", "Delhi", "Bangalore", "Hyderabad",
     "Chennai", "Kolkata", "Pune", "Ahmedabad"],
    size=n
)

device_type = np.random.choice(
    ["mobile", "tablet"],
    size=n,
    p=[0.90, 0.10]
)

is_rooted_device = np.random.choice(
    [0, 1],
    size=n,
    p=[0.95, 0.05]
)

network_type = np.random.choice(
    ["4G", "5G", "WiFi"],
    size=n,
    p=[0.45, 0.35, 0.20]
)

# -----------------------------
# 3. Time-based features
# -----------------------------

timestamp_series = pd.Series(timestamp)

hour = timestamp_series.dt.hour

time_of_day = pd.cut(
    hour,
    bins=[-1, 5, 11, 17, 23],
    labels=["night", "morning", "afternoon", "evening"]
).astype(str)

# -----------------------------
# 4. Create fraud probability
# -----------------------------

# Start with a low base probability
fraud_score = np.full(n, 0.03)

# High transaction amount
fraud_score += np.where(amount > 30000, 0.15, 0)
fraud_score += np.where(amount > 70000, 0.20, 0)

# Rooted device
fraud_score += np.where(is_rooted_device == 1, 0.25, 0)

# Night-time transactions
fraud_score += np.where(time_of_day == "night", 0.12, 0)

# Merchant payments
fraud_score += np.where(
    transaction_type == "merchant_payment",
    0.04,
    0
)

# Tablet usage
fraud_score += np.where(
    device_type == "tablet",
    0.05,
    0
)

# Random variation
fraud_score += np.random.normal(0, 0.02, n)

# Keep probability between 0 and 0.95
fraud_score = np.clip(fraud_score, 0.01, 0.95)

# Generate fraud labels
is_fraud = np.random.binomial(1, fraud_score)

# -----------------------------
# 5. Create final DataFrame
# -----------------------------

df = pd.DataFrame({
    "transaction_id": transaction_id,
    "timestamp": timestamp,
    "user_id": user_id,
    "receiver_id": receiver_id,
    "amount": amount,
    "transaction_type": transaction_type,
    "location": location,
    "device_type": device_type,
    "is_rooted_device": is_rooted_device,
    "network_type": network_type,
    "time_of_day": time_of_day,
    "is_fraud": is_fraud
})

# -----------------------------
# 6. Save dataset
# -----------------------------

df.to_csv("../data/upi_fraud_data.csv", index=False)

print("Dataset generated successfully!")
print("Shape:", df.shape)
print("\nFraud distribution:")
print(df["is_fraud"].value_counts())
print("\nFraud percentage:")
print(df["is_fraud"].mean() * 100)

Dataset generated successfully!
Shape: (5000, 12)

Fraud distribution:
is_fraud
0    4539
1     461
Name: count, dtype: int64

Fraud percentage:
9.22
